# The join that returned more rows than the table

Two tables in `FROM` without a condition is the Cartesian product.

In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE product  (id INTEGER PRIMARY KEY, name TEXT, price REAL);
CREATE TABLE purchase (id INTEGER PRIMARY KEY, product_id INTEGER, units INTEGER);
INSERT INTO product VALUES (1, 'Milk', 1.20), (2, 'Tea', 3.50);
INSERT INTO purchase VALUES (1,1,1),(2,2,2),(3,1,3),(4,2,1),(5,1,2),(6,1,1),(7,2,4),(8,1,1);
""")

def count(sql):
    return db.execute(f"SELECT COUNT(*) FROM ({sql})").fetchone()[0]

print("purchases:", count("SELECT * FROM purchase"), " products:", count("SELECT * FROM product"))
print("no condition:  ", count("SELECT * FROM purchase, product"))

purchases: 8  products: 2
no condition:   16


## And the report doubled

In [2]:
print(db.execute("SELECT SUM(units * price) FROM purchase, product").fetchone()[0], "<- every purchase paired with both prices")
print(db.execute("SELECT SUM(units * price) FROM purchase JOIN product ON purchase.product_id = product.id").fetchone()[0], "<- correct")

70.5 <- every purchase paired with both prices
34.1 <- correct


## Explicit joins fail loudly

In [3]:
print("with WHERE:      ", count("SELECT * FROM purchase, product WHERE purchase.product_id = product.id"))
print("with JOIN ... ON:", count("SELECT * FROM purchase JOIN product ON purchase.product_id = product.id"))
try:
    db.execute("SELECT * FROM purchase JOIN product ON").fetchall()    # a forgotten ON is a syntax error
except sqlite3.OperationalError as e:
    print("forgotten ON:     syntax error ->", e)

with WHERE:       8
with JOIN ... ON: 8
forgotten ON:     syntax error -> incomplete input


The habit: print the row count before and after every join. A join can be
silently wrong only if nobody looks.

**What we used:** tables as sets of tuples, selection, and joins with and
without a condition (session 11, sections 9, 11 and 12).